# Setup

In [ ]:
%%capture
%pip install --quiet pydantic-ai sentence-transformers numpy mcp nest_asyncio openai
# Patch asyncio for Jupyter (REQUIRED — run this before any agent call)
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# Set your OpenRouter API key
OPENROUTER_API_KEY = ''    # e.g. 'sk-or-v1-...'

import os, getpass
if OPENROUTER_API_KEY:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
elif 'OPENROUTER_API_KEY' not in os.environ:
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

# Extracting information without structured outputs


In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

abstract = """
In this study, we introduce a novel deep learning approach for predicting protein-protein interactions (PPIs) in Saccharomyces cerevisiae.
Our method leverages graph neural networks to capture complex molecular interactions and achieves an AUC-ROC score of 0.92 on the independent test set.
The model outperforms traditional machine learning methods and provides interpretable insights into key interacting residues.
"""

resp = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Extract the title, author, organism, method, and metric from the provided abstract.",
        },
        {"role": "user", "content": abstract},
    ]
)

raw_response = resp.choices[0].message.content

print(raw_response) # Hard to parse and to work with!


In [ ]:
lines = raw_response.split('\n')
metric_line = next((line for line in lines if line.startswith('Metric:')), None)

if metric_line:
    parsed_metric = metric_line.replace('Metric:', '').strip()
    print(f"Parsed Metric from raw response: {parsed_metric}")
else:
    print("Metric not found in raw response.")

# Structured output with the raw completions API

In [ ]:
from openai import OpenAI
import os
import json

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

abstract = """
In this study, we introduce a novel deep learning approach for predicting protein-protein interactions (PPIs) in Saccharomyces cerevisiae. 
Our method leverages graph neural networks to capture complex molecular interactions and achieves an AUC-ROC score of 0.92 on the independent test set. 
The model outperforms traditional machine learning methods and provides interpretable insights into key interacting residues.
"""

resp = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Return JSON with keys: title, author, organism, method, metric.", #We have to specify the keys we want to extract from the abstract
        },
        {"role": "user", "content": abstract},
    ],
    response_format={"type": "json_object"},
)

raw = resp.choices[0].message.content
data = json.loads(raw)

print(f"Title: {data.get("title")}")
print(f"Author: {data.get("author")}")
print(f"Organism: {data.get("organism")}")
print(f"Method: {data.get("method")}")
print(f"Metric: {data.get("metric")}")
print(data)


In [ ]:
# We may have intended a numeric value
baseline = 0.5
data.get("metric") - baseline 

In [ ]:
# We might get different data types than we expect
author = data.get("author")
if author:
    print(f'This paper was authored by {author}')
else:
    # Some handling for missing data
    print("No author information available.")
    pass


# Structured output with PydanticAI

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

MODEL = OpenAIChatModel(
    'anthropic/claude-haiku-4.5',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
    ),
)

In [ ]:
from pydantic_ai import Agent
from pydantic import BaseModel

class PaperSummary(BaseModel):
    title: str | None
    author: str | None
    organism: str | None
    method: str | None
    metric: float | None

agent = Agent(model=MODEL, system_prompt="Extract relevant fields from the provided abstract.", output_type=PaperSummary)
result = agent.run_sync(f"Abstract to parse: {abstract}")
structured_output = result.output
print(type(structured_output))
print(structured_output.metric) #Actually returns a float number

In [ ]:
print(structured_output.model_dump_json(indent=2))
# Output can still be hallucinated or output UNKNOWN instead of None

# Exercise

In [ ]:
#TODO create an LLM-call that will use an abstract to extract: biomedical problem addressed, ml approach, input data, and model output
text = """
In this study, we propose iDNA-ABF, a multi-scale deep biological language learning model
that enables the interpretable prediction of DNA methylations based on genomic sequences only.
Benchmarking comparisons show that iDNA-ABF outperforms state-of-the-art methods for
different methylation predictions. By integrating an interpretable analysis mechanism, the model
helps map important sequential determinants to downstream biological functions.
"""
